# VisionLab — Full-Dataset Training on Google Colab

This notebook runs the same `src/` pipeline and `scripts/train_baseline.py`
used locally (see `notebooks/06_first_real_training_run.ipynb`), but on
Colab's GPU and against the **full** datasets instead of the small
stratified subsets used for the local smoke test.

**Why Colab:** Sprint 6's local run (RTX 3050, 4GB VRAM) used only 0.61% of
mushroom's real training data per class due to time constraints. A Colab
GPU (typically a T4 with ~15GB VRAM) gives more headroom, and moving the
long-running training off a laptop that has to stay awake and plugged in
for hours is worth the setup cost.

**What this notebook does, in order:**
1. Check the assigned GPU
2. Clone the repo and install dependencies
3. Authenticate with Kaggle and download the full mushroom + flower datasets
4. Mount Google Drive so checkpoints / TensorBoard logs / MLflow tracking
   survive past this Colab session (Colab's local disk is wiped on
   disconnect — this is *not* optional, skipping it means losing all
   progress if the session drops)
5. Verify the pipeline against the full datasets
6. Launch training (cells provided, **not executed automatically** — run
   them yourself when ready)

**Before running:** this notebook does not start training on its own.
Read through Sections 1–5 first, and decide the epoch count / which
dataset to run in Section 6 yourself.

## 1. Check the GPU

Colab's free tier usually assigns a T4 (~15GB VRAM) — more headroom than
the local RTX 3050 (4GB), but not guaranteed; Colab can also assign no GPU
at all if none is available. If `torch.cuda.is_available()` is `False`
here, go to **Runtime → Change runtime type → GPU** before continuing.

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 2. Clone the Repo and Install Dependencies

Uses the `dev` branch (where all Sprint 1–6 work lives). Colab already
ships a CUDA-enabled PyTorch, so `pip install -e .` should mostly just add
the packages Colab doesn't already have (`mlflow`, `pyyaml`, etc.) rather
than reinstalling `torch`/`torchvision`.

In [ ]:
!git clone -b dev https://github.com/sumeyyesaray/VisionLab.git
%cd VisionLab

In [ ]:
!pip install -q -e .

## 3. Kaggle Authentication and Dataset Download

**Preferred: Colab secrets.** In the left sidebar, click the key icon
("Secrets"), add `KAGGLE_USERNAME` and `KAGGLE_KEY` (from
kaggle.com → your account → Settings → API → "Create New Token", which
downloads a `kaggle.json` containing both values), and toggle "Notebook
access" on for each. The cell below reads them from there.

**Fallback:** if secrets aren't set, it'll prompt you to upload
`kaggle.json` directly instead.

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab secrets.")
except Exception:
    from google.colab import files
    print("Colab secrets not found — upload your kaggle.json:")
    uploaded = files.upload()
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "kaggle.json").write_bytes(list(uploaded.values())[0])
    (kaggle_dir / "kaggle.json").chmod(0o600)
    print("kaggle.json saved.")

Downloads straight into the layout `src/data/loaders.py` already expects
— both zips extract to exactly `dataset_root` in `configs/mushroom.yaml` /
`configs/flower.yaml`, no reorganizing needed (verified against the local
copy before writing this notebook: mushroom's zip root is
`train.csv`/`val.csv`/`test.csv` + `merged_dataset/`, flower's is a single
`flower_data/` folder).

In [ ]:
!pip install -q kaggle
!mkdir -p datasets/extracted/mushroom datasets/extracted
!kaggle datasets download -d zlatan599/mushroom1 -p datasets/extracted/mushroom --unzip
!kaggle datasets download -d waseemalastal/the-oxford-flowers-102-dataset -p datasets/extracted --unzip

In [ ]:
!echo "mushroom root:" && ls datasets/extracted/mushroom
!echo "flower root:" && ls datasets/extracted/flower_data

## 4. Mount Google Drive for Persistent Results

Colab's local disk is wiped when the session disconnects — checkpoints,
TensorBoard logs, and the MLflow database all need to live on Drive
instead, or a dropped connection loses everything. This symlinks
`outputs/` and `runs/` to Drive-backed folders and points
`MLFLOW_TRACKING_URI` (read by `src/training/tracking.py`) at a Drive path
too.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/VisionLab_results")
(DRIVE_ROOT / "outputs" / "checkpoints").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "runs").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "mlflow").mkdir(parents=True, exist_ok=True)

!rm -rf outputs runs
!ln -s "{DRIVE_ROOT}/outputs" outputs
!ln -s "{DRIVE_ROOT}/runs" runs

os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{DRIVE_ROOT}/mlflow/mlflow.db"

print(f"Persisting to: {DRIVE_ROOT}")
print("If this session disconnects mid-training, the last checkpoint will",
      "still be there — resume with --resume outputs/checkpoints/<file>.pt")

## 5. Verify the Pipeline Against the Full Datasets

Reuses `scripts/check_pipeline.py` as-is (see Sprint 3) — no Colab-specific
code needed here. Expect **689,520 / 15,616 / 15,614** for mushroom and
**6,552 / 818 / 819** for flower (the same numbers from Sprint 1's EDA);
if these don't match, the Kaggle download didn't land in the structure the
pipeline expects — stop and check section 3 before training anything.

In [ ]:
!python scripts/check_pipeline.py

## 6. Training

**Not run automatically — run the cells below yourself.** Unlike Sprint
6's local smoke test, there's no `--subset-per-class` here, so this trains
on the full dataset. A few things worth deciding before running:

- **Epoch count:** `configs/*.yaml` currently default to 6 (tuned for the
  local smoke test). Override with `--epochs N` — start with something
  modest (e.g. 10–15) and check the first couple of epochs' timing before
  committing to more; see the note on session limits below.
- **`num_workers`:** the committed configs use `4`. Check
  `os.cpu_count()` in this Colab instance (`!nproc`) and adjust
  `configs/mushroom.yaml`/`configs/flower.yaml` if it's noticeably
  different — this was flagged as an unresolved empirical question back in
  Sprint 3 (`docs/hyperparameters.md`).
- **Session limits:** free Colab disconnects after ~90 minutes idle (no
  cell activity) or at a hard ~12 hour ceiling. Because checkpoints now
  save to Drive every epoch (section 4), a dropped session doesn't lose
  progress — reconnect and add `--resume outputs/checkpoints/<dataset>_resnet50.pt`
  to the same command to continue from the last completed epoch.
- **Do not conclude anything about mushroom's `light` augmentation preset
  from a short run here.** Sprint 6's notebook already covers why: that
  choice assumes the full dataset's abundance (4,080 images/class), which
  *is* satisfied now that we're not subsetting — so this run is the actual
  test of that assumption, not a repeat of the smoke test's artifact.

In [ ]:
!nproc

In [ ]:
# Mushroom — full dataset (689,520 train images)
!python scripts/train_baseline.py --config configs/mushroom.yaml --epochs 15

In [ ]:
# Flower — full dataset (6,552 train images)
!python scripts/train_baseline.py --config configs/flower.yaml --epochs 15

## 7. Monitoring While Training Runs

TensorBoard renders inline in Colab. Run this in a **separate cell above
the training cell** (or in a second browser tab pointed at this same
notebook) so it updates live while training runs in the cell below it —
Colab executes cells sequentially, so a cell placed *after* a
long-running `!python` cell won't render until that cell finishes.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

MLflow's data is being written to Drive
(`/content/drive/MyDrive/VisionLab_results/mlflow/mlflow.db`) but Colab
doesn't have a clean inline `mlflow ui` the way TensorBoard does. Easiest
options: sync Google Drive to your local machine (Drive for Desktop) and
run `mlflow ui --backend-store-uri sqlite:///path/to/mlflow.db` locally
the same way as in Sprint 6, or copy the `.db` file down via the Colab
file browser after a run finishes.

## 8. After Training

- Checkpoints, TensorBoard logs, and the MLflow database are all on Drive
  under `VisionLab_results/` — nothing here needs a manual export step.
- Pull the final checkpoint back into the local repo's
  `outputs/checkpoints/` if the next step (evaluation, Error Analysis) runs
  locally instead of in Colab.
- Compare these full-dataset numbers against Sprint 6's subset numbers
  (flower: 92.3% val acc; mushroom: 62.0% val acc) — mushroom in
  particular should look very different now that the "artificially scarce
  subset" explanation from Sprint 6 no longer applies.